# Video Clone Douyin - Google Colab GPU Backend

Notebook này khởi chạy backend FastAPI của **Video Clone Douyin** trên GPU Google Colab và xuất URL public qua Cloudflare Tunnel.

Cách dùng:
1. Chọn **Runtime > Change runtime type > GPU**.
2. Chạy lần lượt các cell bên dưới (cell 3 sẽ mount Google Drive để lưu file xuất).
3. Sao chép URL `https://...trycloudflare.com`.
4. Mở app desktop, vào **Cài đặt > Google Colab GPU**, dán URL và bấm **Kiểm tra** rồi chọn **Remote Google Colab**.
5. Khi xuất **MP4 / Audio / SRT** trong app, tệp được lưu vào **Google Drive > VideoCloneExports** (không tải về máy tính). App hiện thông báo kèm link Drive để mở xem.
6. Tải **model dịch** (TranslateGemma, Qwen 3, …) trong app desktop: **Cấu hình → Model dịch** — model tải lên cache Colab GPU và hiện **đúng tên** trong **Công cụ dịch** ở sidebar.


In [ ]:
#@title 1. Cấu hình repo và kiểm tra GPU
import os, subprocess, textwrap

# Repo Colab runtime riêng, chỉ chứa backend cần thiết để tránh đưa toàn bộ code desktop lên Colab.
REPO_URL = "https://github.com/nqthaivl/videocolab.git"  #@param {type:"string"}
PROJECT_DIR = "/content/videocolab"
PORT = 3900

print("GPU Colab:")
!nvidia-smi || true

if not os.path.isdir(PROJECT_DIR):
    !git clone "$REPO_URL" "$PROJECT_DIR"
else:
    print("Repo đã tồn tại, cập nhật code mới nhất...")
    %cd "$PROJECT_DIR"
    !git pull --ff-only || true

%cd "$PROJECT_DIR"
print("Thư mục dự án:", os.getcwd())


In [ ]:
#@title 2. Cai dependencies backend
import os, subprocess, sys

print("Cai system packages...")
!apt-get update -qq
!apt-get install -y -qq ffmpeg git wget

print("Cai uv va Python dependencies tu pyproject.toml...")
!python -m pip install -U pip uv
!uv pip install --system -e .

def disable_broken_torchvision(reason):
    print("torchvision cua Colab bi lech voi torch, go bo vi Video Clone khong can torchvision.")
    print(reason)
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"])
    for name in list(sys.modules):
        if name.startswith("torchvision"):
            sys.modules.pop(name, None)
    os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"

print("Kiem tra torchvision tuy chon cua Colab...")
try:
    import torchvision  # noqa: F401
    print("torchvision OK.")
except Exception as exc:
    disable_broken_torchvision(exc)

print("Kiem tra transformers co HiggsAudioV2TokenizerModel...")
try:
    from transformers import HiggsAudioV2TokenizerModel  # noqa: F401
    print("transformers OK: HiggsAudioV2TokenizerModel san sang.")
except Exception as exc:
    if "torchvision" in str(exc) or "_HAS_OPS" in str(exc):
        disable_broken_torchvision(exc)
    print("transformers thieu HiggsAudioV2TokenizerModel, cai lai ban tuong thich...")
    print(exc)
    subprocess.check_call(["uv", "pip", "install", "--system", "--reinstall", "transformers>=5.3.0"])
    from transformers import HiggsAudioV2TokenizerModel  # noqa: F401
    print("transformers da duoc sua.")

print("Tai cloudflared...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

print("Hoan tat cai dat.")


In [ ]:
#@title 3. Mount Google Drive, khởi chạy backend và Cloudflare Tunnel
import os, re, subprocess, time, requests, signal

from google.colab import drive

DATA_DIR = "/content/video-clone-data"
CACHE_DIR = "/content/video-clone-models"
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/VideoCloneExports"
DRIVE_FOLDER_URL = "https://drive.google.com/drive/my-drive"

print("Mount Google Drive (bắt buộc để xuất MP4/Audio/SRT lên Drive)...")
drive.mount("/content/drive")
os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)

try:
    from google.colab import auth
    from googleapiclient.discovery import build

    auth.authenticate_user()
    service = build("drive", "v3")
    query = (
        "mimeType='application/vnd.google-apps.folder' and "
        "name='VideoCloneExports' and trashed=false"
    )
    results = service.files().list(q=query, fields="files(id, webViewLink)").execute()
    folders = results.get("files", [])
    if folders:
        folder_id = folders[0]["id"]
        DRIVE_FOLDER_URL = folders[0].get("webViewLink") or f"https://drive.google.com/drive/folders/{folder_id}"
    else:
        created = service.files().create(
            body={"name": "VideoCloneExports", "mimeType": "application/vnd.google-apps.folder"},
            fields="id, webViewLink",
        ).execute()
        folder_id = created["id"]
        DRIVE_FOLDER_URL = created.get("webViewLink") or f"https://drive.google.com/drive/folders/{folder_id}"
    print("Thư mục xuất Drive:", DRIVE_EXPORT_DIR)
    print("Link thư mục Drive:", DRIVE_FOLDER_URL)
except Exception as exc:
    print("Không lấy được link thư mục Drive qua API, dùng My Drive mặc định:", exc)
    print("Thư mục xuất Drive:", DRIVE_EXPORT_DIR)

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

env = {
    **os.environ,
    "PYTHONUNBUFFERED": "1",
    "OMNIVOICE_DATA_DIR": DATA_DIR,
    "OMNIVOICE_CACHE_DIR": CACHE_DIR,
    "HF_HOME": CACHE_DIR,
    "HF_HUB_CACHE": CACHE_DIR,
    "TORCH_HOME": CACHE_DIR,
    "OMNIVOICE_MCP_DISABLE": "1",
    "OMNIVOICE_UI_PORT": "5174",
    "OMNIVOICE_ALLOWED_ORIGINS": "http://127.0.0.1:5174,http://localhost:5174,null",
    "VIDEO_DUBBING_DISABLE_MODEL_PRELOAD": "1",
    "OMNIVOICE_COLAB_RUNTIME": "1",
    "OMNIVOICE_ACTIVATED": "1",
    "OMNIVOICE_ASR_BACKEND": "faster-whisper",
    "ASR_MODEL_FASTER": "Systran/faster-whisper-large-v3",
    "TRANSFORMERS_NO_TORCHVISION": "1",
    "HF_HUB_DISABLE_XET": "1",
    "OMNIVOICE_DRIVE_EXPORT_DIR": DRIVE_EXPORT_DIR,
    "OMNIVOICE_DRIVE_FOLDER_URL": DRIVE_FOLDER_URL,
}

backend_cmd = ["python", "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", str(PORT)]
print("Khởi chạy backend Video Clone...")
backend = subprocess.Popen(backend_cmd, cwd=os.path.join(PROJECT_DIR, "backend"), env=env)

health_url = f"http://127.0.0.1:{PORT}/health"
for i in range(120):
    try:
        r = requests.get(health_url, timeout=2)
        if r.ok:
            print("Backend sẵn sàng:", r.json())
            try:
                drive_status = requests.get(f"http://127.0.0.1:{PORT}/colab/drive-status", timeout=3).json()
                print("Drive export:", drive_status)
            except Exception as exc:
                print("Không kiểm tra được trạng thái Drive export:", exc)
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Backend không phản hồi /health sau 120 giây")

print("Khởi tạo Cloudflare Tunnel...")
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", health_url.rsplit("/", 1)[0]],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

try:
    public_url = None
    while True:
        line = tunnel.stdout.readline()
        if not line:
            time.sleep(0.2)
            continue
        if "trycloudflare.com" in line or "error" in line.lower():
            print("[cloudflared]", line.strip())
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match and public_url is None:
            public_url = match.group(0)
            print("\n" + "=" * 72)
            print("VIDEO CLONE COLAB BACKEND ĐÃ SẴN SÀNG")
            print("URL API Colab:")
            print(public_url)
            print("\nDán URL này vào app: Cài đặt > Google Colab GPU > URL API Colab")
            print("Xuất MP4/Audio/SRT trong app sẽ lưu vào Google Drive > VideoCloneExports")
            print("Link thư mục Drive:", DRIVE_FOLDER_URL)
            print("=" * 72 + "\n")
except KeyboardInterrupt:
    print("Dừng backend và tunnel...")
finally:
    tunnel.terminate()
    backend.terminate()
